# Trace Validation — T1.2

Visual confirmation of the **TRD §1.3 invariant** before anything downstream is built:

- **(a) aggregate requests/minute** must look flat / noisy-flat throughout — *no visible bump at the shift*.
- **(b) per-archetype share** must show a visible ramp at `shift_start_min`, with `agentic_tool_using` rising and `short_conversational` falling.

If (a) shows a bump at the shift, STOP and fix `src/trace_gen/generator.py` before continuing to Phase 2.

Run `python -m src.trace_gen.generator` first to produce `data/trace.csv`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import ARCHETYPES, TRACE_DEFAULTS

DATA_DIR = REPO_ROOT / "data"
SHIFT_START = int(TRACE_DEFAULTS["shift_start_min"])
SHIFT_END = SHIFT_START + int(TRACE_DEFAULTS["shift_duration_min"])

df = pd.read_csv(DATA_DIR / "trace.csv")
print(f"{len(df)} requests over {df['minute'].max() + 1} minutes")
df.head()

## (a) Aggregate requests/minute — must be flat

In [ ]:
per_minute = df.groupby("minute").size()
ratio = per_minute.std() / per_minute.mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(per_minute.index, per_minute.values, color="#334155", lw=1.4, label="requests/min")
ax.axhline(per_minute.mean(), color="#94a3b8", ls=":", lw=1.2, label=f"mean = {per_minute.mean():.1f}")
ax.axvline(SHIFT_START, color="#dc2626", ls="--", lw=1.5, label=f"shift_start_min = {SHIFT_START}")
ax.axvspan(SHIFT_START, SHIFT_END, color="#dc2626", alpha=0.07)
ax.set_xlabel("simulation minute")
ax.set_ylabel("requests / minute")
ax.set_title(f"(a) Aggregate arrival rate — std/mean = {ratio:.4f} (gate: < 0.3)")
ax.set_ylim(0, per_minute.max() * 1.25)
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig(DATA_DIR / "trace_aggregate_rate.png", dpi=150)
plt.show()

print(f"std/mean = {ratio:.4f} -> {'PASS' if ratio < 0.3 else 'FAIL'}")
before = per_minute.loc[:SHIFT_START - 1].mean()
after = per_minute.loc[SHIFT_END:].mean()
print(f"mean before shift = {before:.2f}, after = {after:.2f}, relative gap = {abs(before - after) / before:.4f}")

## (b) Per-archetype share — must ramp at the shift

In [ ]:
shares = (
    df.groupby(["minute", "true_archetype"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=ARCHETYPES, fill_value=0)
)
shares = shares.div(shares.sum(axis=1), axis=0)
smoothed = shares.rolling(window=5, min_periods=1, center=True).mean()

colors = {
    "short_conversational": "#2563eb",
    "long_context_rag": "#7c3aed",
    "agentic_tool_using": "#dc2626",
    "batch_offline": "#059669",
}

fig, ax = plt.subplots(figsize=(11, 4.5))
for archetype in ARCHETYPES:
    ax.plot(shares.index, shares[archetype], color=colors[archetype], alpha=0.22, lw=1)
    ax.plot(smoothed.index, smoothed[archetype], color=colors[archetype], lw=2, label=archetype)
ax.axvline(SHIFT_START, color="#dc2626", ls="--", lw=1.5)
ax.axvspan(SHIFT_START, SHIFT_END, color="#dc2626", alpha=0.07)
ax.annotate("compositional shift", xy=(SHIFT_START, 0.62), xytext=(SHIFT_START + 6, 0.66), fontsize=9, color="#dc2626")
ax.set_xlabel("simulation minute")
ax.set_ylabel("share of requests")
ax.set_title("(b) Archetype composition — the shift is visible ONLY here")
ax.set_ylim(0, 0.72)
ax.legend(loc="upper right", fontsize=9, ncol=2)
fig.tight_layout()
fig.savefig(DATA_DIR / "trace_archetype_shares.png", dpi=150)
plt.show()

## Verdict

In [ ]:
pre = df[df["minute"] < SHIFT_START]["true_archetype"].value_counts(normalize=True)
post = df[df["minute"] >= SHIFT_END]["true_archetype"].value_counts(normalize=True)
summary = pd.DataFrame({"pre_shift_share": pre, "post_shift_share": post}).reindex(ARCHETYPES)
summary["delta"] = summary["post_shift_share"] - summary["pre_shift_share"]
print(summary.round(4).to_string())

aggregate_flat = ratio < 0.3 and abs(before - after) / before < 0.1
composition_shifted = summary.loc["agentic_tool_using", "delta"] > 0.2

print()
print(f"(a) aggregate flat, no bump at shift : {'PASS' if aggregate_flat else 'FAIL'}")
print(f"(b) composition ramps at shift       : {'PASS' if composition_shifted else 'FAIL'}")
print()
print("T1.2 gate:", "PASS - safe to proceed to Phase 2" if aggregate_flat and composition_shifted else "FAIL - fix T1.1 first")